# AgentNet (Ubuntu subset) exploratory data analysis

**Goal.** Understand the Ubuntu part of AgentNet (5,000 tasks) before deciding how to use
it for training: what each field means, which values are missing, and how trajectories look.

**Input.** `data/raw/agentnet/agentnet_ubuntu_5k.jsonl` and `meta_data_merged.jsonl`,
fetched by `scripts/download_agentnet.py`. The dataset revision is pinned in
`data/raw/agentnet/REVISION.txt`. Screenshots are not downloaded.
The Windows and macOS file is excluded for now.

**Conclusion.** Of 5,000 Ubuntu tasks, 2,673 are scored by the judge and 2,327 are not. The quality fields are
null together and the share varies by domain, so scored tasks are not a random sample. Among scored
tasks 86.6% are judged completed. The agent's own `success` status is wrong for 165 of 2,488
scored tasks (6.6%), so it is not a reliable label. Step labels are sparse: `last_step_redundant`
is missing on 41.3% of steps, `reflection` on 44.5% and `marks` on 93.5%. Where present,
`last_step_correct` is true on 94.5% of steps. The metadata file adds nothing for Ubuntu.

In [1]:
import sys

if sys.version_info[:2] != (3, 14):
    raise RuntimeError(
        "Python 3.14 is required. Run `uv python install 3.14` and `uv sync`, "
        "then open this notebook with `uv run jupyter lab notebooks/`."
    )

In [2]:
import plotly.express as px
import polars as pl
from IPython.display import Markdown, display

from agentnet import (
    DATA_DIR,
    REVISION_FILE,
    SUBSET_FILES,
    scan_jsonl,
    step_table,
    task_table,
)

In [3]:
Markdown(f"Dataset revision: `{REVISION_FILE.read_text().strip()}`")

Dataset revision: `xlangai/AgentNet@d76ee50a63fad81cfdbe576416757d7c2091ed50`

## Field guide

The dataset card documents only some fields and gives no score scales or missing value
rules. The column "Basis" separates the card's wording from what was measured here.

| Field | Meaning | Basis |
|---|---|---|
| `task_id` | Unique id, a timestamp followed by a UUID | measured: unique across all 5,000 rows |
| `instruction` | Human annotated task description | card |
| `natural_language_task` | More natural rewrite of the task, synthesized by a summarizer | card |
| `actual_task` | Detailed task specification, synthesized by a summarizer | card |
| `domain` | Application or category of the task, 12 values | measured, equals `domains` in the metadata |
| `task_completed` | Whether the trajectory solved the task, judged automatically | card says "quality metric"; judge inferred from `reason` |
| `alignment_score` | Judge score for how well the steps match the task, integers 0 to 10 | range measured |
| `efficiency_score` | Judge score for how efficient the steps are, integers 1 to 10 | range measured |
| `task_difficulty` | Judge score for difficulty, integers 0 to 10 | range measured |
| `reason` | Free text justification for the scores | inferred from content, not on the card |
| `traj` | List of steps | card |
| `traj.index` | Zero based step position | measured |
| `traj.image` | Screenshot filename taken before the action | card example |
| `traj.step_id` | Step identifier | not documented |
| `traj.marks` | List of strings attached to a step | not documented, mostly missing |
| `value.observation` | Generated description of the visual scene | card |
| `value.thought` | Reasoning and planning text | card |
| `value.action` | Natural language description of the action | card |
| `value.code` | Executable `pyautogui` or `computer` call; x and y are fractions of the screen in [0, 1] | card for code, range measured |
| `value.last_step_correct` | Whether the previous step was correct | card says "current step"; index 0 is mostly labeled, so semantics unclear |
| `value.last_step_redundant` | Whether the previous step was redundant | card, same caveat |
| `value.reflection` | Error analysis produced by a reflector model | card, mostly missing |

Metadata file (`meta_data_merged.jsonl`): for Ubuntu rows only `task_id`, `instruction`,
`verify_feedback.quality`, `domains`, `system` and `step_num` are filled. Everything else
in that file is only populated for Windows and macOS (checked at the end).

In [4]:
trajectories = scan_jsonl(DATA_DIR / SUBSET_FILES["ubuntu"])
tasks = (
    task_table(trajectories)
    .with_columns(pl.col("task_completed").is_not_null().alias("scored"))
    .collect()
)

## Schema and missing values per task column

In [5]:
schema = trajectories.collect_schema()
display(Markdown(f"Raw file also contains `traj`: `{schema['traj']}`"))
pl.DataFrame(
    {
        "column": tasks.columns,
        "dtype": [str(tasks.schema[column]) for column in tasks.columns],
        "nulls": [tasks[column].null_count() for column in tasks.columns],
    }
).with_columns((pl.col("nulls") / tasks.height).alias("null_rate"))

Raw file also contains `traj`: `List(Struct({'index': Int64, 'step_id': String, 'image': String, 'value': Struct({'code': String, 'last_step_correct': Boolean, 'last_step_redundant': Boolean, 'observation': String, 'thought': String, 'action': String, 'reflection': String}), 'marks': List(String)}))`

column,dtype,nulls,null_rate
str,str,i64,f64
"""task_id""","""String""",0,0.0
"""instruction""","""String""",0,0.0
"""task_completed""","""Boolean""",2327,0.4654
"""alignment_score""","""Int64""",2327,0.4654
"""efficiency_score""","""Int64""",2327,0.4654
…,…,…,…
"""natural_language_task""","""String""",2331,0.4662
"""actual_task""","""String""",2331,0.4662
"""domain""","""String""",0,0.0


## Unscored tasks

The four quality fields (`task_completed`, `alignment_score`, `efficiency_score`,
`task_difficulty`) and `reason` are null together, and `natural_language_task` and
`actual_task` are null on almost the same rows. Nothing in the card explains this, so these
rows are treated as **unscored**: the quality judge was apparently never run on them. The
share depends strongly on the domain, so scored rows are not a random sample of tasks.

In [6]:
px.bar(
    tasks.group_by("domain")
    .agg(pl.len().alias("tasks"), (1 - pl.col("scored").mean()).alias("unscored_rate"))
    .sort("unscored_rate"),
    x="unscored_rate",
    y="domain",
    orientation="h",
    hover_data=["tasks"],
    labels={"unscored_rate": "Share of unscored tasks", "domain": "Domain", "tasks": "Tasks"},
    title="Share of unscored tasks per domain",
)

### Null pattern consistency

In [7]:
tasks.select(
    pl.col("scored").sum().alias("scored_rows"),
    (~pl.col("scored")).sum().alias("unscored_rows"),
    (pl.col("alignment_score").is_null() == pl.col("scored")).sum().alias("score_mismatch"),
    (pl.col("reason").is_null() == pl.col("scored")).sum().alias("reason_mismatch"),
    pl.col("natural_language_task").is_null().sum().alias("natural_language_null"),
)

scored_rows,unscored_rows,score_mismatch,reason_mismatch,natural_language_null
u32,u32,u32,u32,u32
2673,2327,0,0,2331


## Overview

In [8]:
tasks.group_by("scored").agg(
    pl.len().alias("tasks"),
    pl.col("task_completed").mean().alias("completed_rate"),
    pl.col("n_steps").mean().alias("mean_steps"),
    pl.col("n_steps").median().alias("median_steps"),
    pl.col("n_steps").max().alias("max_steps"),
)

scored,tasks,completed_rate,mean_steps,median_steps,max_steps
bool,u32,f64,f64,f64,u32
false,2327,null,14.699613,14.0,95
true,2673,0.866442,18.047886,17.0,55


In [9]:
px.histogram(
    tasks,
    x="n_steps",
    color="scored",
    nbins=60,
    barmode="overlay",
    opacity=0.6,
    labels={"n_steps": "Steps per trajectory", "scored": "Scored", "count": "Tasks"},
    title="Trajectory length",
).update_yaxes(title_text="Tasks")

In [10]:
px.bar(
    tasks.group_by("domain").agg(pl.len().alias("tasks")).sort("tasks"),
    x="tasks",
    y="domain",
    orientation="h",
    labels={"tasks": "Tasks", "domain": "Domain"},
    title="Tasks per domain",
)

In [11]:
px.histogram(
    tasks.filter(pl.col("scored"))
    .unpivot(
        ["alignment_score", "efficiency_score", "task_difficulty"],
        index="task_id",
        variable_name="score",
    )
    .sort("value"),
    x="value",
    color="score",
    barmode="group",
    labels={"value": "Score", "score": "Judge score"},
    title="Judge scores for scored tasks",
).update_yaxes(title_text="Tasks")

## Scores by completion (scored tasks only)

In [12]:
tasks.filter(pl.col("scored")).group_by("task_completed").agg(
    pl.len().alias("tasks"),
    pl.col("alignment_score").mean().alias("mean_alignment"),
    pl.col("efficiency_score").mean().alias("mean_efficiency"),
    pl.col("task_difficulty").mean().alias("mean_difficulty"),
    pl.col("n_steps").mean().alias("mean_steps"),
)

task_completed,tasks,mean_alignment,mean_efficiency,mean_difficulty,mean_steps
bool,u32,f64,f64,f64,f64
false,357,5.812325,6.728291,5.148459,11.778711
true,2316,9.745682,7.586788,5.603195,19.014249


In [13]:
px.histogram(
    tasks.with_columns(pl.col("instruction").str.len_chars().alias("instruction_chars")),
    x="instruction_chars",
    nbins=60,
    labels={"instruction_chars": "Instruction length (characters)"},
    title="Instruction length",
).update_yaxes(title_text="Tasks")

In [14]:
steps = (
    step_table(trajectories).collect().join(tasks.select("task_id", "task_completed"), on="task_id")
)

In [15]:
display(Markdown(f"## Step level missing values ({steps.height:,} steps)"))
pl.DataFrame(
    {
        "field": [
            "last_step_correct",
            "last_step_redundant",
            "reflection (null or empty)",
            "marks",
        ],
        "missing_steps": [
            steps["last_step_correct"].null_count(),
            steps["last_step_redundant"].null_count(),
            steps.filter(
                pl.col("reflection_chars").is_null() | (pl.col("reflection_chars") == 0)
            ).height,
            steps["n_marks"].null_count(),
        ],
    }
).with_columns((pl.col("missing_steps") / steps.height).alias("missing_rate"))

## Step level missing values (82,448 steps)

field,missing_steps,missing_rate
str,i64,f64
"""last_step_correct""",10466,0.126941
"""last_step_redundant""",34013,0.412539
"""reflection (null or empty)""",36703,0.445165
"""marks""",77098,0.935111


## Step labels where present

Rates over labeled steps only. A null label is not treated as false.

In [16]:
steps.select(
    pl.col("last_step_correct").mean().alias("correct_rate"),
    pl.col("last_step_redundant").mean().alias("redundant_rate"),
    pl.col("thought_chars").mean().alias("mean_thought_chars"),
    pl.col("observation_chars").mean().alias("mean_observation_chars"),
)

correct_rate,redundant_rate,mean_thought_chars,mean_observation_chars
f64,f64,f64,f64
0.945375,0.070135,1505.052676,1001.071754


In [17]:
px.bar(
    steps.group_by("action_name").agg(pl.len().alias("steps")).sort("steps"),
    x="steps",
    y="action_name",
    orientation="h",
    labels={"steps": "Steps", "action_name": "Action"},
    title="Actions",
)

## Termination status versus `task_completed`

The last step is nearly always `computer.terminate(status=...)`, the agent's own claim of
success or failure. `task_completed` comes from the judge, so disagreement shows how
reliable the agent's self report is. Null `task_completed` marks unscored tasks.

In [18]:
steps.filter(pl.col("action_name") == "computer.terminate").group_by(
    "terminate_status", "task_completed"
).agg(pl.len().alias("tasks")).sort("terminate_status", "task_completed")

terminate_status,task_completed,tasks
str,bool,u32
"""failure""",false,192
"""failure""",true,5
"""success""",null,2321
"""success""",false,165
"""success""",true,2323


In [19]:
px.density_heatmap(
    steps.drop_nulls(["x", "y"]),
    x="x",
    y="y",
    nbinsx=40,
    nbinsy=40,
    color_continuous_scale="Blues",
    labels={"x": "Horizontal position", "y": "Vertical position", "z": "Steps"},
    title="Pointer targets as fractions of screen width and height",
).update_yaxes(autorange="reversed")

In [20]:
meta = scan_jsonl(DATA_DIR / SUBSET_FILES["meta"]).filter(pl.col("system") == "Ubuntu").collect()
joined = tasks.join(meta.select("task_id", "step_num", "domains"), on="task_id")
display(
    Markdown(
        f"## Metadata for Ubuntu rows\n\n{meta.height} rows. `step_num` equals "
        f"`n_steps` on {(joined['step_num'] == joined['n_steps']).mean():.1%} of tasks "
        f"and `domains` equals `domain` on "
        f"{(joined['domains'] == joined['domain']).mean():.1%}, so the metadata file "
        "adds no information for Ubuntu."
    )
)
pl.DataFrame(
    {
        "meta_column": meta.columns,
        "null_rate": [meta[column].null_count() / meta.height for column in meta.columns],
    }
)

## Metadata for Ubuntu rows

5000 rows. `step_num` equals `n_steps` on 100.0% of tasks and `domains` equals `domain` on 100.0%, so the metadata file adds no information for Ubuntu.

meta_column,null_rate
str,f64
"""task_id""",0.0
"""instruction""",0.0
"""verify_feedback""",0.0
"""domains""",0.0
"""system""",0.0
…,…
"""task_description_alignment""",1.0
"""task_description_ambiguity""",1.0
"""complexity""",1.0
